
  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%">


# Summer Institute in Computational Social Sciences - Buenos Aires 2026
# Taller: Procesamiento de Lenguaje Natural y polarización
# Clasificación de tweets con distintas representaciones: TF, TF-IDF y Word Embeddings

# Introducción

El objetivo de este notebook es comparar distintas formas de representar texto como *features* para un modelo de clasificación, usando como caso el dataset **HatEval** (SemEval-2019 Task 5), que ya vienen utilizando en las prácticas de este taller (`data/hateval_train_df.csv`, `data/hateval_dev_df.csv`, `data/hateval_test_df.csv`).

En este ejercicio usamos el split de **test** (`data/hateval_test_df.csv`). Cada fila es un tweet con las siguientes columnas relevantes:

- `id`: identificador del tweet
- `text`: el texto del tweet
- `language`: idioma del tweet (`en` o `es`)
- `HS`: 1 si el tweet contiene discurso de odio (*hate speech*), 0 en caso contrario
- `TR`, `AG`: otras anotaciones (agresividad, si el odio está dirigido a un individuo o a un grupo) que no vamos a usar acá

La tarea de clasificación va a ser predecir `HS` (discurso de odio sí/no) a partir del texto del tweet.

Como en la última sección vamos a usar **embeddings preentrenados en español** (SBWCE), y el dataset tiene tweets tanto en inglés como en español, nos vamos a quedar solamente con los tweets en español (`language == 'es'`) para poder comparar las tres representaciones sobre el mismo conjunto de datos.

Al igual que en el ejemplo con reseñas de Amazon (`cap0/ejemplo_clasificacion.ipynb`), vamos a entrenar en cada caso una regresión logística regularizada por LASSO (penalización L1), variando el hiperparámetro de regularización $C$ (a menor $C$, mayor regularización) y eligiendo el mejor valor mediante validación cruzada.

Vamos a comparar tres formas de vectorizar los tweets:

1. **TF** (*Term Frequency*, bolsa de palabras con conteos)
2. **TF-IDF** (*Term Frequency - Inverse Document Frequency*)
3. **Word embeddings preentrenados** (promedio de vectores de palabras)

In [5]:
## Ejecutar para descargar los embeddings preentrenados en español (SBWCE)
!wget -P ./models https://cs.famaf.unc.edu.ar/~ccardellino/SBWCE/SBW-vectors-300-min5.bin.gz && gunzip ./models/SBW-vectors-300-min5.bin.gz
!pip install gensim
!git clone https://github.com/gefero/factor_data_tuto_NLP_SICSS.git

Cloning into 'factor_data_tuto_NLP_SICSS'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 48 (delta 7), reused 27 (delta 3), pack-reused 15 (from 1)
Receiving objects: 100% (48/48), 63.70 MiB | 24.53 MiB/s, done.
Resolving deltas: 100% (9/9), done.


# TF con LASSO

## Preparación de los datos

Cargamos `data/hateval_test_df.csv`, nos quedamos con los tweets en español y preprocesamos el texto:

1. Convertimos a minúsculas
2. Eliminamos URLs y menciones (`@usuario`), que son ruido propio de los tweets
3. Eliminamos signos de puntuación
4. Reemplazamos números por la palabra `DIGITO`
5. Eliminamos acentos y caracteres no ASCII

`X` va a ser el texto preprocesado y `y` la variable binaria `HS` (discurso de odio).

In [2]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

In [8]:
data_path = './factor_data_tuto_NLP_SICSS/data/hateval_test_df.csv'

In [9]:
# Cargamos los datos
tweets = pd.read_csv(data_path)

# Nos quedamos con los tweets en español (los embeddings preentrenados que
# usaremos más adelante están entrenados en español)
tweets = tweets[tweets['language'] == 'es'].reset_index(drop=True)

# Función para preprocesar el texto
def preprocess_text(text):
    # Convertir a minúsculas
    text = text.lower()

    # Eliminar URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Eliminar menciones (@usuario)
    text = re.sub(r'@\w+', ' ', text)

    # Reemplazar puntuación
    text = re.sub(r'[^\w\s]', ' ', text)

    # Reemplazar números por 'DIGITO'
    text = re.sub(r'\d+', 'DIGITO', text)

    # Reemplazar caracteres no ASCII (acentos, etc.)
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

    # Colapsar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Aplicamos el preprocesamiento
tweets['text_clean'] = tweets['text'].apply(preprocess_text)

# Preparamos las variables para el modelo
X = tweets['text_clean']
y = tweets['HS']

## División de datos y búsqueda de hiperparámetros

Dividimos los datos en entrenamiento (75%) y prueba (25%), manteniendo la proporción de clases (`stratify=y`).

Armamos un pipeline con:

1. **Vectorización TF**: `CountVectorizer` con ngramas de 1 y 2 palabras
2. **Regresión logística LASSO** (penalización L1, solver `liblinear`)

Y buscamos el mejor valor de `C` (inverso de la fuerza de regularización) probando 30 valores en escala logarítmica entre $10^{-10}$ y $10^{1}$, evaluando cada uno con validación cruzada de 5 particiones usando ROC AUC.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=664,
    stratify=y
)

# Creamos una grilla de valores para el parámetro C (inverso de la penalización)
C_values = np.logspace(-10, 1, 30)

# Pipeline con TF y LASSO
pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Configuramos la validación cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=234)

# Almacenaremos los resultados aquí
cv_results = []

In [11]:
%%time
# Para cada valor de C
for C in C_values:
    pipeline_tf.set_params(classifier__C=C)

    # Calculamos el ROC AUC con validación cruzada
    scores = cross_val_score(
        pipeline_tf,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    cv_results.append({
        'C': C,
        'mean_roc_auc': scores.mean(),
        'std_roc_auc': scores.std()
    })

# Convertimos resultados a DataFrame
cv_results_df = pd.DataFrame(cv_results)

# Encontramos el mejor C
best_idx = cv_results_df['mean_roc_auc'].idxmax()
best_C_tf = cv_results_df.loc[best_idx, 'C']

print(f"Mejor valor de C: {best_C_tf:.6f}")
print(f"Mejor ROC AUC (CV): {cv_results_df.loc[best_idx, 'mean_roc_auc']:.3f} ± {cv_results_df.loc[best_idx, 'std_roc_auc']:.3f}")

Mejor valor de C: 0.727895
Mejor ROC AUC (CV): 0.771 ± 0.015
CPU times: user 15.3 s, sys: 604 µs, total: 15.3 s
Wall time: 15.4 s


## Modelo final

Entrenamos el pipeline final con el mejor valor de `C` encontrado y evaluamos sobre el conjunto de prueba con ROC AUC, accuracy, precision, recall y F1.

In [13]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

final_pipeline_tf.fit(X_train, y_train)

# Predicciones en el conjunto de prueba
y_pred = final_pipeline_tf.predict(X_test)
y_pred_proba = final_pipeline_tf.predict_proba(X_test)[:, 1]

# Métricas finales
results_tf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

for metric, value in results_tf.items():
    print(f"{metric}: {value:.3f}")

roc_auc: 0.816
accuracy: 0.762
precision: 0.765
recall: 0.612
f1: 0.680


# TF-IDF con LASSO

A diferencia de TF (que solo cuenta ocurrencias), **TF-IDF** pondera cada término según qué tan frecuente es dentro de un tweet (TF) pero penalizando los términos que aparecen en muchos tweets distintos (IDF). Esto suele darle más peso a palabras discriminativas y menos a palabras muy comunes.

Reutilizamos el mismo split (`X_train`, `X_test`, `y_train`, `y_test`) obtenido en la sección anterior para que la comparación sea directa, y repetimos el mismo procedimiento de búsqueda de `C` pero con `TfidfVectorizer` en lugar de `CountVectorizer`.

In [ ]:
# Pipeline con TF-IDF y LASSO
pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

cv_results = []

In [ ]:
%%time
for C in C_values:
    pipeline_tfidf.set_params(classifier__C=C)

    scores = cross_val_score(
        pipeline_tfidf,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    cv_results.append({
        'C': C,
        'mean_roc_auc': scores.mean(),
        'std_roc_auc': scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)

best_idx = cv_results_df['mean_roc_auc'].idxmax()
best_C_tfidf = cv_results_df.loc[best_idx, 'C']

print(f"Mejor valor de C: {best_C_tfidf:.6f}")
print(f"Mejor ROC AUC (CV): {cv_results_df.loc[best_idx, 'mean_roc_auc']:.3f} ± {cv_results_df.loc[best_idx, 'std_roc_auc']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tfidf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

final_pipeline_tfidf.fit(X_train, y_train)

y_pred = final_pipeline_tfidf.predict(X_test)
y_pred_proba = final_pipeline_tfidf.predict_proba(X_test)[:, 1]

results_tfidf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

for metric, value in results_tfidf.items():
    print(f"{metric}: {value:.3f}")

# Word embeddings como features

## Idea general

En lugar de representar cada tweet como un vector disperso de conteos (TF) o pesos (TF-IDF) sobre el vocabulario, ahora vamos a representarlo como el **promedio de los vectores de embedding preentrenados** de sus palabras. Usamos los embeddings estáticos en español **SBWCE** (`SBW-vectors-300-min5`, ya descargados al inicio del notebook), donde cada palabra se representa con un vector denso de 300 dimensiones.

El preprocesamiento acá es más simple: solo eliminamos URLs y menciones, y reemplazamos números por `DIGITO`. No hace falta sacar puntuación ni acentos porque las palabras que no estén en el vocabulario de los embeddings simplemente se ignoran (*out-of-vocabulary*).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
from gensim.models import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Descargas necesarias para tokenización
nltk.download('punkt_tab')
nltk.download('punkt')

# Cargamos los datos y nos quedamos con los tweets en español
tweets = pd.read_csv(data_path)
tweets = tweets[tweets['language'] == 'es'].reset_index(drop=True)

# Preprocesamiento simple: solo removemos URLs, menciones y reemplazamos dígitos
def preprocess_text_embed(text):
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\d+', 'DIGITO', text)
    return text

tweets['text_clean'] = tweets['text'].apply(preprocess_text_embed)

# Cargamos el modelo de word embeddings
def load_embeddings(path):
    print("Cargando embeddings...")
    return KeyedVectors.load_word2vec_format(path, binary=True)

word_vectors = load_embeddings("./models/SBW-vectors-300-min5.bin")

# Función para obtener el vector promedio de un tweet
def get_mean_vector(text, word_vectors, vector_size=300):
    words = word_tokenize(text.lower())  # Tokenizamos y convertimos a minúsculas
    word_vectors_list = []

    for word in words:
        try:
            vector = word_vectors[word]
            word_vectors_list.append(vector)
        except KeyError:
            continue  # Ignoramos palabras que no están en el embedding

    if word_vectors_list:
        return np.mean(word_vectors_list, axis=0)
    else:
        return np.zeros(vector_size)  # Vector de ceros si no hay palabras válidas

# Convertimos los tweets a vectores
print("Vectorizando tweets...")
tweet_vectors = [get_mean_vector(text, word_vectors) for text in tweets['text_clean']]

# Convertimos a DataFrame
X_embed = pd.DataFrame(tweet_vectors, columns=[f'V{i+1}' for i in range(300)])
X_embed['id'] = tweets['id']
y_embed = tweets['HS']

In [ ]:
X_embed.head()

## División de datos y búsqueda de hiperparámetros

Repetimos el mismo esquema: split 75/25 estratificado (mismo `random_state` que antes, para que el split quede alineado con el de las secciones de TF y TF-IDF) y búsqueda del mejor `C` por validación cruzada, pero ahora sobre los vectores de embeddings en lugar del texto.

In [ ]:
# División en train y test
X_train_e, X_test_e, y_train, y_test = train_test_split(
    X_embed, y_embed,
    test_size=0.25,
    random_state=664,
    stratify=y_embed
)

# Separamos el id
train_ids = X_train_e['id']
test_ids = X_test_e['id']
X_train_embed = X_train_e.drop('id', axis=1)
X_test_embed = X_test_e.drop('id', axis=1)

# Grilla de valores para C
C_values = np.logspace(-10, 1, 30)

# Configuramos la validación cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=234)

cv_results = []

In [ ]:
%%time
print("Realizando validación cruzada...")
for C in C_values:
    model = LogisticRegression(
        C=C,
        penalty='l1',
        solver='liblinear',
        random_state=234
    )

    scores = cross_val_score(
        model,
        X_train_embed,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    cv_results.append({
        'C': C,
        'mean_roc_auc': scores.mean(),
        'std_roc_auc': scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)

best_idx = cv_results_df['mean_roc_auc'].idxmax()
best_C_embed = cv_results_df.loc[best_idx, 'C']

print(f"\nMejor valor de C: {best_C_embed:.6f}")
print(f"Mejor ROC AUC (CV): {cv_results_df.loc[best_idx, 'mean_roc_auc']:.3f} ± {cv_results_df.loc[best_idx, 'std_roc_auc']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_model_embed = LogisticRegression(
    C=best_C_embed,
    penalty='l1',
    solver='liblinear',
    random_state=234
)

final_model_embed.fit(X_train_embed, y_train)

y_pred = final_model_embed.predict(X_test_embed)
y_pred_proba = final_model_embed.predict_proba(X_test_embed)[:, 1]

results_embed = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("\nResultados finales:")
for metric, value in results_embed.items():
    print(f"{metric}: {value:.3f}")

# Comparación de resultados

In [ ]:
results_tf_df = pd.DataFrame([results_tf], index=['TF'])
results_tfidf_df = pd.DataFrame([results_tfidf], index=['TF-IDF'])
results_embed_df = pd.DataFrame([results_embed], index=['Word Embeddings'])

comparison_df = pd.concat([results_tf_df, results_tfidf_df, results_embed_df])

print("Comparación de resultados de los modelos:")
display(comparison_df)

# Ejercicio

Entrenar y tunear un Random Forest usando features construidas con TF-IDF y otro con embeddings, para clasificar los tweets en español del dataset HatEval.

¿Cuál resulta más eficiente? ¿Por qué?

Como desafío adicional: ¿qué esperarías que pase si usás estos mismos embeddings en español (SBWCE) para vectorizar los tweets en **inglés** del dataset (`language == 'en'`)?

In [ ]:
###